In [23]:
# LOAD PARAMETERS FROM TRAINED MODEL
import json
import re
from collections import Counter
import numpy as np
import pandas as pd

PARAMS_PATH = "logreg_params.npz"
TFIDF_META_PATH = "logreg_tfidf.json"
OUTPUT_PATH = "logreg_for_stack.npz"


def fill_missing_text(X_df, columns):
    for col in columns:
        X_df[col] = X_df[col].fillna("")


def build_text_specs(tfidf_meta):
    return [
        (col, {k: int(v) for k, v in tfidf_meta[col]["vocab"].items()}, np.asarray(tfidf_meta[col]["idf"], dtype=np.float64))
        for col in ["feel_text", "food", "soundtrack"]
    ]


def load_logreg_params(params_path=PARAMS_PATH, tfidf_meta_path=TFIDF_META_PATH):
    params = np.load(params_path, allow_pickle=True)
    with open(tfidf_meta_path, "r", encoding="utf-8") as f:
        tfidf_meta = json.load(f)

    loaded = {
        "coef": np.asarray(params["coef"], dtype=np.float64),
        "intercept": np.asarray(params["intercept"], dtype=np.float64).ravel(),
        "class_order": params["class_order"].tolist(),
        "numeric_cols": params["numeric_cols"].tolist(),
        "num_means": params["num_means"].astype(np.float64),
        "num_stds": params["num_stds"].astype(np.float64),
    }
    loaded["num_stds"] = np.where(np.abs(loaded["num_stds"]) < 1e-12, 1.0, loaded["num_stds"])
    loaded["text_specs"] = build_text_specs(tfidf_meta)
    return loaded


df = pd.read_csv("cleaned_dataset.csv")
X = df.drop(["target", "id", "painting"], axis=1).copy()

fill_missing_text(X, ["feel_text", "food", "soundtrack"])

loaded = load_logreg_params()
coef = loaded["coef"]
intercept = loaded["intercept"]
class_order = loaded["class_order"]
numeric_cols = loaded["numeric_cols"]
num_means = loaded["num_means"]
num_stds = loaded["num_stds"]
TEXT_SPECS = loaded["text_specs"]


In [24]:
# REBUILD TF-IDF BLOCKS AND PREDICTION ARRAYS
TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")


def build_tfidf(text_series, vocab, idf_vec):
    block = np.zeros((len(text_series), len(vocab)), dtype=np.float64)

    for i, text in enumerate(text_series):
        tokens = TOKEN_RE.findall(str(text).lower())
        if not tokens:
            continue
        counts = Counter(tok for tok in tokens if tok in vocab)
        if not counts:
            continue

        total = float(sum(counts.values()))
        norm_sq = 0.0

        for tok, count in counts.items():
            j = vocab[tok]
            val = (count / total) * idf_vec[j]
            block[i, j] = val
            norm_sq += val * val

        norm = np.sqrt(norm_sq)
        if norm > 0.0:
            block[i, :] /= norm

    return block


def build_logreg_features(X_df):
    X_num = X_df[numeric_cols].to_numpy(dtype=np.float64)
    X_num = np.where(np.isnan(X_num), num_means, X_num)
    X_num = (X_num - num_means) / num_stds

    text_blocks = [
        build_tfidf(X_df[col], vocab, idf_vec)
        for col, vocab, idf_vec in TEXT_SPECS
    ]

    return np.hstack([X_num] + text_blocks)


def predict(X_mat, coef_mat, intercept_vec):
    if coef_mat.ndim == 1 or (coef_mat.ndim == 2 and coef_mat.shape[0] == 1):
        logits = np.asarray(X_mat @ coef_mat.ravel() + float(intercept_vec[0])).ravel()
        logits = np.clip(logits, -50.0, 50.0)
        p_pos = 1.0 / (1.0 + np.exp(-logits))
        return np.column_stack([1.0 - p_pos, p_pos])

    logits = np.asarray(X_mat @ coef_mat.T) + intercept_vec.reshape(1, -1)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


In [25]:
# PREDICT AND SAVE FOR NEW DATA

def build_stack_payload(proba, class_labels, row_idx):
    return {
        "logreg_proba": np.asarray(proba, dtype=np.float64),
        "logreg_class_order": np.asarray(class_labels, dtype=object),
        "row_indices": np.asarray(row_idx),
    }

X_tx = build_logreg_features(X)
logreg_proba = predict(X_tx, coef, intercept)
logreg_class_order = list(class_order)
row_indices = X.index.to_numpy()

logreg_stack_payload = build_stack_payload(
    logreg_proba,
    logreg_class_order,
    row_indices,
)

np.savez(OUTPUT_PATH, **logreg_stack_payload)

print("Prediction arrays created.")
print("logreg_proba shape:", logreg_proba.shape)
print("Payload keys:", list(logreg_stack_payload.keys()))
print(f"Saved: {OUTPUT_PATH}")


Prediction arrays created.
logreg_proba shape: (1610, 3)
Payload keys: ['logreg_proba', 'logreg_class_order', 'row_indices']
Saved: logreg_for_stack.npz
